In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

catalog_name="ecommerce"

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_order_items")
display(df.limit(5))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,1,2000000028279,1,GBP,11,10%,2,web,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,2,2000000377445,1,GBP,5,4%,1,web,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 09:44:32,CUST000000239553,643612,1,2000000417639,4,INR,4871,8%,3245,web,FEST20,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,1,2000000422664,Two,AED,74,0%,18,app,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,2,2000000238159,1,AED,1012,11%,109,app,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z


In [0]:
df.printSchema()

root
 |-- dt: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- item_seq: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price_currency: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



In [0]:
df_silver=df.dropDuplicates(["order_id","item_seq"])

In [0]:
df_silver=df_silver.withColumn("quantity" ,
                               F.when(F.col("quantity")== "Two" , 2)\
                               .otherwise(F.col("quantity")).cast("int")    
)

df_silver=df_silver.withColumn("discount_pct" , F.regexp_replace(F.col("discount_pct") , "%" , "").cast("double"))

df_silver=df_silver.withColumn("unit_price" , F.regexp_replace(F.col("unit_price") , "[$]" , "").cast("double"))\
                   .withColumn("coupon_code" , F.lower(F.trim(F.col("coupon_code"))))\
                   .withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")))     





In [0]:
display(df_silver.limit(10))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2,Website,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 22:53:52,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1,Website,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 09:44:32,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245,Website,fest20,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,3,2000000094205,2,AED,229.0,15.0,20,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 04:45:03,CUST000000175269,643613,4,2000000293790,2,AED,2222.0,0.0,801,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 21:09:10,CUST000000117094,643614,1,2000000271415,3,INR,252.0,10.0,82,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 13:03:57,CUST000000016826,643615,1,2000000050980,1,GBP,71.0,5.0,9,Website,save50,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z
2025-08-01,2025-08-01 17:54:51,CUST000000252796,643616,1,2000000463254,1,USD,6.0,9.0,1,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z


In [0]:
df_silver=df_silver.withColumn("dt" , F.to_date("dt", "yyyy-MM-dd"))\
                   .withColumn("order_ts",
    F.coalesce(
        F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"),  
        F.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")      
    )).withColumn(
    "item_seq",
    F.col("item_seq").cast("int"))\
    .withColumn(
    "tax_amount",
    F.regexp_replace("tax_amount", r"[^0-9.\-]", "").cast("double"))\
    .withColumn(
    "processed_time", F.current_timestamp())

In [0]:
display(df_silver.limit(10))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,3,2000000094205,2,AED,229.0,15.0,20.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,4,2000000293790,2,AED,2222.0,0.0,801.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T21:09:10.000Z,CUST000000117094,643614,1,2000000271415,3,INR,252.0,10.0,82.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T13:03:57.000Z,CUST000000016826,643615,1,2000000050980,1,GBP,71.0,5.0,9.0,Website,save50,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z
2025-08-01,2025-08-01T17:54:51.000Z,CUST000000252796,643616,1,2000000463254,1,USD,6.0,9.0,1.0,Mobile,null,order_items_2025-08-01.csv,2026-02-10T10:55:35.938Z,2026-02-10T11:39:31.071Z


In [0]:
df_silver.printSchema()

root
 |-- dt: date (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- item_seq: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price_currency: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- processed_time: timestamp (nullable = false)



In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")